In [1]:
import torch
from run_epoch import get_vit_model, get_loaders
from torch.profiler import schedule
from torch.profiler import profile, ProfilerActivity, record_function


model = get_vit_model()
train_loader, val_loader = get_loaders()

Train Data: 4318
Val Data: 1080


In [2]:
def trace_handler(p):
    # Выводим табличку в консоль
    output = p.key_averages().table(sort_by="self_cuda_time_total", row_limit=10)
    print(output)
    
    # Сохраняем JSON файл для Chrome Tracing
    # p.step_num - это текущий глобальный шаг обучения, он поможет не перезаписать файлы от разных циклов
    p.export_chrome_trace("./traces/model_inference_trace_" + str(p.step_num) + ".json")

In [3]:
train_loader_iter, val_loader_iter = iter(train_loader), iter(val_loader)

In [4]:
sheduler = schedule(skip_first=3, wait=2, warmup=1, active=2, repeat=1)

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    schedule=sheduler,
    on_trace_ready=trace_handler
) as prof:
    for _ in range(100):
        
        train_data = next(train_loader_iter)
        with record_function("inference"):
            res = model(train_data[0].to("cuda:0"))
        
        prof.step()

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                              inference         0.00%       0.000us         0.00%       0.000us       0.000us      41.464ms        54.49%      41.464ms      20.732ms             2  
                                            aten::copy_         9.30%     463.057ms         9.74%     485.067ms      18.110us      22.028ms        28.95%      22.028ms       0.822us         26784  
         

StopIteration: 

In [1]:
import torch
from run_epoch import get_vit_model, get_loaders
from torch.profiler import schedule
from torch.profiler import profile, ProfilerActivity, record_function


model = get_vit_model()
train_loader, val_loader = get_loaders()
train_loader_iter, val_loader_iter = iter(train_loader), iter(val_loader)

Train Data: 4318
Val Data: 1080


In [4]:
from profiler import Profile, CustomSchedule

sheduler = CustomSchedule(skip_first=3, wait=2, warmup=1, active=2)
with Profile(model, name="usa",) as prof:
    for _ in range(20):
        train_data = next(train_loader_iter)
        output = model(train_data[0].to("cuda:0"))

        prof.step()

StopIteration: 

In [6]:
prof.to_perfetto()

Trace saved to trace.json


In [5]:
prof.summary()

Summary:
{'name': 'Rearrange', 'cat': 'forward', 'ph': 'X', 'ts': 26380147797313.707, 'dur': 284.58982706069946, 'pid': 3912226, 'tid': 0}
{'name': 'Linear', 'cat': 'forward', 'ph': 'X', 'ts': 26380147797632.83, 'dur': 779.8485457897186, 'pid': 3912226, 'tid': 0}
{'name': 'Dropout', 'cat': 'forward', 'ph': 'X', 'ts': 26380147798674.97, 'dur': 73.93583655357361, 'pid': 3912226, 'tid': 0}
{'name': 'Linear', 'cat': 'forward', 'ph': 'X', 'ts': 26380147798821.473, 'dur': 143.58758926391602, 'pid': 3912226, 'tid': 0}
{'name': 'Linear', 'cat': 'forward', 'ph': 'X', 'ts': 26380147799027.562, 'dur': 120.65097689628601, 'pid': 3912226, 'tid': 0}
{'name': 'Linear', 'cat': 'forward', 'ph': 'X', 'ts': 26380147799200.496, 'dur': 122.61047959327698, 'pid': 3912226, 'tid': 0}
{'name': 'Softmax', 'cat': 'forward', 'ph': 'X', 'ts': 26380147799483.72, 'dur': 77.64995098114014, 'pid': 3912226, 'tid': 0}
{'name': 'Dropout', 'cat': 'forward', 'ph': 'X', 'ts': 26380147799611.184, 'dur': 42.08087921142578, 'p

In [3]:
prof.summary()

Summary:
{'name': 'Rearrange', 'cat': 'forward', 'ph': 'X', 'ts': 26380037926331.137, 'dur': 5667.880177497864, 'pid': 3912226, 'tid': 0}
{'name': 'Linear', 'cat': 'forward', 'ph': 'X', 'ts': 26380037932061.492, 'dur': 35746.06031179428, 'pid': 3912226, 'tid': 0}
{'name': 'Dropout', 'cat': 'forward', 'ph': 'X', 'ts': 26380037971971.5, 'dur': 113.73311281204224, 'pid': 3912226, 'tid': 0}
{'name': 'Linear', 'cat': 'forward', 'ph': 'X', 'ts': 26380037972170.305, 'dur': 345.5840051174164, 'pid': 3912226, 'tid': 0}
{'name': 'Linear', 'cat': 'forward', 'ph': 'X', 'ts': 26380037972576.2, 'dur': 250.60772895812988, 'pid': 3912226, 'tid': 0}
{'name': 'Linear', 'cat': 'forward', 'ph': 'X', 'ts': 26380037972885.29, 'dur': 215.18394351005554, 'pid': 3912226, 'tid': 0}
{'name': 'Softmax', 'cat': 'forward', 'ph': 'X', 'ts': 26380037977281.754, 'dur': 10120.783001184464, 'pid': 3912226, 'tid': 0}
{'name': 'Dropout', 'cat': 'forward', 'ph': 'X', 'ts': 26380037987469.75, 'dur': 53.43183875083923, 'pid'